# 06 — Exercises: Data Handling and Processing

This notebook contains hands-on exercises covering data loading, cleaning, EDA, feature engineering, and outlier detection.

**What you'll learn:**
- How to load, inspect, and understand datasets before doing any analysis
- How to build a systematic data cleaning pipeline to fix real-world data issues
- How to perform exploratory data analysis (EDA) to uncover patterns and relationships
- How to engineer new features that make data more useful for machine learning
- How to detect and treat outliers using statistical methods

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')
%matplotlib inline

**Why these imports?**
- **pandas** — the core library for tabular data manipulation (DataFrames)
- **numpy** — numerical computing; used for math operations, random numbers, and handling `NaN`
- **matplotlib** — the foundational plotting library; gives fine-grained control over charts
- **seaborn** — built on top of matplotlib; provides beautiful statistical visualizations with less code
- `sns.set_style('darkgrid')` sets a clean background grid for all plots
- `%matplotlib inline` tells Jupyter to render plots directly inside the notebook

---
## Exercise 1: Data Loading and Inspection

**Objective:** Practice loading data from multiple formats and performing initial inspection.

**Why is this important?**

Before doing any analysis or building any model, you **must** understand your data. Loading and inspecting data is always the first step. It helps you answer critical questions:
- How big is the dataset? (rows × columns)
- What types of data do we have? (numbers, text, dates, categories)
- Is there missing data? How much?
- Are there any obvious issues like wrong data types?

Think of it like a doctor examining a patient before prescribing treatment — you need a diagnosis first.

### Task 1: Load the Titanic dataset and display the first 10 rows

In [ ]:
df = sns.load_dataset("titanic")
df.head(10)

**Explanation:**

`sns.load_dataset("titanic")` loads the classic Titanic dataset directly from seaborn's built-in datasets. This is the same dataset used in the famous Kaggle competition — it contains passenger information like age, sex, class, fare, and whether they survived.

`df.head(10)` shows the **first 10 rows**. This is your first look at the data — you want to check:
- Do the column names make sense?
- Do the values look reasonable?
- Are there any obvious `NaN` (missing) values visible?

> **Tip:** Always start with `.head()` — it's the quickest way to get a feel for your data.

### Task 2: Print the shape, data types, and memory usage

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nMemory Usage:")
df.info(memory_usage='deep')

**Explanation:**

- **`df.shape`** returns `(rows, columns)` — e.g., `(891, 15)` means 891 passengers and 15 features. This tells you the scale of your data.
- **`df.dtypes`** shows each column's data type:
  - `int64` / `float64` = numeric (integers / decimals)
  - `object` = usually strings/text
  - `category` = categorical data (limited set of values)
  - `bool` = True/False
- **`df.info(memory_usage='deep')`** gives a comprehensive summary including non-null counts and actual memory consumption. The `'deep'` option calculates true memory usage for object columns (which store strings of varying length).

> **Why check data types?** Wrong types cause bugs. For example, if a numeric column is stored as `object` (string), you can't do math on it until you convert it.

### Task 3: Use `df.describe(include="all")` and interpret the output

In [ ]:
df.describe(include='all')

**Interpreting `df.describe(include='all')`:**

This is one of the most powerful one-liners in Pandas. It generates summary statistics for every column:

**For numeric columns:**
| Statistic | What it tells you |
|-----------|------------------|
| `count` | Number of non-null values (helps spot missing data) |
| `mean` | Average value |
| `std` | Standard deviation — how spread out the values are |
| `min` / `max` | Range of values — look for impossible values (e.g., negative ages) |
| `25%`, `50%`, `75%` | Quartiles — the 50% is the median. If mean ≠ median, the data is skewed |

**For categorical columns:**
| Statistic | What it tells you |
|-----------|------------------|
| `count` | Number of non-null values |
| `unique` | How many distinct values exist |
| `top` | The most frequent value (mode) |
| `freq` | How often the most frequent value appears |

> **Pro tip:** Compare `count` across columns — any column with a count less than the total rows has missing values.

### Task 4: Identify columns with missing values and calculate missingness percentage

In [ ]:
print(f"Missing Values:\n{df.isnull().sum()}")
print(f"\nMissing Percentages:\n{(df.isnull().sum() / len(df) * 100).round(2)}")

**Explanation:**

- **`df.isnull().sum()`** counts `NaN` values per column. In the Titanic dataset, you'll typically find missing values in `age`, `deck`, and `embarked`.
- **Missingness percentage** `(missing / total × 100)` is more informative than raw counts — a column with 70% missing data might need to be dropped entirely, while 2% missing can usually be imputed.

**Rules of thumb for handling missing data:**
- **< 5% missing:** Safe to impute (fill with mean/median/mode) or drop rows
- **5–30% missing:** Imputation is preferred; be careful about which method you choose
- **> 30% missing:** Consider dropping the column entirely, unless it's critical

> **Key insight:** Missing data is rarely random. In the Titanic dataset, `deck` is mostly missing because lower-class passengers didn't have recorded deck assignments — the missingness itself tells a story.

---
## Exercise 2: Data Cleaning Pipeline

**Objective:** Build a complete data cleaning pipeline for a messy dataset.

**Why is this important?**

Real-world data is **never** clean. It comes with inconsistent formatting, missing values, duplicates, invalid entries, and mixed data types. A data cleaning pipeline is a systematic, step-by-step process to fix these issues. The order matters:

1. **Fix text/formatting issues first** (so deduplication works correctly)
2. **Handle invalid values** (so statistics like median are meaningful)
3. **Impute missing values** (using the now-correct data)
4. **Remove duplicates** (after names are normalized)
5. **Validate** (confirm everything looks right)

> **"Garbage in, garbage out"** — if you feed dirty data into a model, you'll get unreliable results no matter how good the algorithm is.

### Task 1: Create the messy dataset

In [ ]:
messy_data = pd.DataFrame({
    "Name": ["  Alice ", "Bob", "CHARLIE", "alice", "David", "Bob", None],
    "Age": [25, -3, 30, 25, 200, 35, 28],
    "Email": ["alice@mail.com", "bob@", "charlie@mail.com", "alice@mail.com", "david@mail.com", "bob@mail.com", "eve@mail.com"],
    "Salary": ["50000", "60K", "70000", None, "80000", "60000", "55000"],
    "Date_Joined": ["2020-01-15", "2020/02/20", "March 3, 2020", "2020-01-15", "2020-04-10", "2020-02-20", "2020-05-01"]
})

print("Original messy data:")
messy_data

**Spot the problems in this dataset:**

| Column | Issues |
|--------|--------|
| `Name` | Leading/trailing whitespace (`"  Alice "`), inconsistent casing (`"CHARLIE"` vs `"alice"`), `None` value |
| `Age` | Impossible values: `-3` (negative) and `200` (no human is 200 years old) |
| `Email` | Invalid format: `"bob@"` has no domain |
| `Salary` | Mixed types: `"60K"` has a letter, `None` is missing entirely — all stored as strings |
| `Date_Joined` | Three different date formats: `"2020-01-15"`, `"2020/02/20"`, `"March 3, 2020"` |

These are **extremely common** problems in real data. Let's fix them one by one.

### Task 2: Clean the Name column — strip whitespace, normalize to title case, handle None

In [ ]:
messy_data["Name"] = messy_data["Name"].str.strip().str.title()
messy_data["Name"].fillna("Unknown", inplace=True)

print("After cleaning Name column:")
messy_data["Name"]

**Explanation:**

- **`.str.strip()`** removes leading and trailing whitespace. `"  Alice "` becomes `"Alice"`.
- **`.str.title()`** converts to title case: `"CHARLIE"` → `"Charlie"`, `"alice"` → `"Alice"`. This ensures consistent formatting so that `"alice"` and `"Alice"` are recognized as the same name.
- **`.fillna("Unknown")`** replaces `None`/`NaN` with a placeholder string.

> **Why title case?** Normalization is essential for deduplication. Without it, `"Alice"` and `"alice"` would be treated as different people.

### Task 3: Fix the Age column — replace impossible values with NaN, impute with median

In [ ]:
messy_data.loc[(messy_data["Age"] < 0) | (messy_data["Age"] > 120), "Age"] = np.nan
messy_data["Age"].fillna(messy_data["Age"].median(), inplace=True)

print("After fixing Age column:")
messy_data["Age"]

**Explanation:**

- **`.loc[]` with a boolean condition** selects rows where Age < 0 OR Age > 120, and sets those values to `NaN`. This is called **domain-based validation** — we know human ages must fall within a reasonable range.
- **`.fillna(median)`** replaces the `NaN` values with the median age. 

**Why median instead of mean?**
- The **mean** is sensitive to extreme values (outliers pull it up or down)
- The **median** is the middle value and is robust to outliers
- For skewed data or data with outliers, median is almost always the better choice for imputation

> **Example:** Ages = [25, 28, 30, 35]. Mean = 29.5, Median = 29. Both are reasonable here since there are no outliers after we removed the invalid values.

### Task 4: Clean the Salary column — remove non-numeric characters, convert to float, impute missing

In [ ]:
messy_data["Salary"] = messy_data["Salary"].str.replace(r"[^\d.]", "", regex=True)
messy_data["Salary"] = pd.to_numeric(messy_data["Salary"], errors="coerce")
messy_data["Salary"].fillna(messy_data["Salary"].median(), inplace=True)

print("After cleaning Salary column:")
messy_data["Salary"]

**Explanation:**

This is a 3-step process to fix the Salary column:

1. **`.str.replace(r"[^\d.]", "", regex=True)`** — Uses a regex (regular expression) to remove everything that isn't a digit (`\d`) or a decimal point (`.`). So `"60K"` becomes `"60"` (the "K" is stripped out).

2. **`pd.to_numeric(..., errors="coerce")`** — Converts strings to numbers. The `errors="coerce"` parameter is crucial: if a value can't be converted, it becomes `NaN` instead of throwing an error.

3. **`.fillna(median)`** — Fills missing salaries with the median value.

> **Note:** The `"60K"` becomes `60`, not `60000`. In a real scenario, you'd want smarter parsing (detecting "K" = thousands, "M" = millions). This exercise keeps it simple to focus on the technique.

### Task 5: Standardize the Date_Joined column to a consistent datetime format

In [ ]:
messy_data["Date_Joined"] = pd.to_datetime(messy_data["Date_Joined"], format="mixed")

print("After standardizing Date_Joined column:")
messy_data["Date_Joined"]

**Explanation:**

**`pd.to_datetime(..., format="mixed")`** is a powerful Pandas function that can parse multiple date formats automatically. It handles:
- `"2020-01-15"` (ISO format)
- `"2020/02/20"` (slash-separated)
- `"March 3, 2020"` (written-out month)

The `format="mixed"` parameter tells Pandas to try multiple parsers for each value. The result is a uniform `datetime64` type, which enables:
- Date arithmetic (e.g., days between two dates)
- Extracting components (year, month, day of week)
- Time-based filtering and sorting

> **Why standardize dates?** Inconsistent date formats are one of the most common data quality issues. Without standardization, you can't sort chronologically or calculate time differences.

### Task 6: Remove duplicate rows based on Name and Email

In [ ]:
print(f"Rows before deduplication: {len(messy_data)}")
messy_data.drop_duplicates(subset=["Name", "Email"], keep="first", inplace=True)
print(f"Rows after deduplication: {len(messy_data)}")

**Explanation:**

**`drop_duplicates(subset=["Name", "Email"], keep="first")`** removes rows that have the same Name AND Email combination.

- **`subset`** — which columns to check for duplicates (not all columns need to match)
- **`keep="first"`** — keeps the first occurrence, removes subsequent duplicates
- **`inplace=True`** — modifies the DataFrame directly instead of returning a copy

> **Why we cleaned names first:** If we hadn't normalized `"  Alice "` and `"alice"` to `"Alice"`, they would appear as different records and deduplication wouldn't catch them.

### Task 7: Validate the Email column using a simple regex pattern

In [ ]:
messy_data["Valid_Email"] = messy_data["Email"].str.contains(r"^[\w.]+@[\w]+\.[\w]+$", regex=True)

print("Final cleaned dataset:")
messy_data

**Explanation:**

The regex pattern **`^[\w.]+@[\w]+\.[\w]+$`** validates email format:
- `^` — start of string
- `[\w.]+` — one or more word characters or dots (the local part, e.g., `alice`)
- `@` — literal @ symbol
- `[\w]+` — one or more word characters (the domain, e.g., `mail`)
- `\.` — literal dot
- `[\w]+$` — one or more word characters until end (the TLD, e.g., `com`)

So `"bob@"` would fail because there's nothing after the `@`.

> **Note:** This is a simplified regex. Real email validation is far more complex (RFC 5322 allows many edge cases). For production use, consider a dedicated validation library. This is sufficient for basic data quality checks.

---
## Exercise 3: Exploratory Data Analysis

**Objective:** Conduct a thorough EDA on a real-world dataset.

**Why is this important?**

EDA is the process of **understanding your data through statistics and visualization** before applying any machine learning. It helps you:

- **Discover patterns** — e.g., do people tip more on weekends?
- **Identify relationships** — e.g., is there a correlation between bill size and tip?
- **Spot anomalies** — e.g., are there unusually large tips?
- **Guide feature engineering** — knowing relationships helps you create better features
- **Choose the right model** — the distribution of your data affects which algorithms work best

EDA is not a one-time step — you'll come back to it repeatedly as you build understanding.

### Task 1: Load the tips dataset

In [ ]:
tips = sns.load_dataset("tips")
tips.head()

**About the Tips dataset:**

This dataset contains 244 restaurant bills with columns:
- `total_bill` — the total bill amount in dollars
- `tip` — the tip amount in dollars
- `sex` — gender of the bill payer
- `smoker` — whether the party included smokers
- `day` — day of the week (Thur, Fri, Sat, Sun)
- `time` — Lunch or Dinner
- `size` — party size (number of people)

It's a great dataset for EDA because it has a mix of numeric and categorical variables with interesting relationships to explore.

### Task 2: Answer analytical questions using Pandas

In [ ]:
# Average tip by day of the week
print("Average tip by day:")
print(tips.groupby("day")["tip"].mean().round(2))

# Which gender tips more on average (as percentage of total bill)
tips["tip_pct"] = (tips["tip"] / tips["total_bill"] * 100).round(2)
print("\nAverage tip percentage by gender:")
print(tips.groupby("sex")["tip_pct"].mean().round(2))

# Correlation between total bill and tip
print(f"\nCorrelation between total_bill and tip: {tips['total_bill'].corr(tips['tip']):.4f}")

# Distribution of party sizes
print("\nParty size distribution:")
print(tips["size"].value_counts().sort_index())

**Explanation of the analysis:**

1. **`groupby("day")["tip"].mean()`** — Groups all rows by day, then calculates the average tip for each day. This is a **split-apply-combine** operation: split data into groups, apply a function (mean), combine results.

2. **Tip percentage** `(tip / total_bill × 100)` — Raw tip amounts are misleading because larger bills naturally have larger tips. Tip *percentage* is a fairer comparison. We create a new column `tip_pct` to store this.

3. **`.corr()`** — Calculates the Pearson correlation coefficient (r) between two variables:
   - `r = 1.0` → perfect positive correlation
   - `r = 0.0` → no linear relationship
   - `r = -1.0` → perfect negative correlation
   - A value around 0.67 for total_bill vs tip means a **strong positive correlation** — bigger bills get bigger tips.

4. **`.value_counts().sort_index()`** — Counts how many times each party size appears, sorted by size. This reveals that parties of 2 are by far the most common.

### Task 3: Create visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram of total_bill with KDE overlay
axes[0, 0].hist(tips["total_bill"], bins=20, color="#FF6B6B", alpha=0.7, edgecolor="black", density=True)
tips["total_bill"].plot.kde(ax=axes[0, 0], color="darkred", linewidth=2)
axes[0, 0].set_title("Distribution of Total Bill", fontweight="bold")
axes[0, 0].set_xlabel("Total Bill ($)")
axes[0, 0].set_ylabel("Density")

# Box plot of tip grouped by day
sns.boxplot(x="day", y="tip", data=tips, ax=axes[0, 1], palette="Set2")
axes[0, 1].set_title("Tips by Day of Week", fontweight="bold")
axes[0, 1].set_xlabel("Day")
axes[0, 1].set_ylabel("Tip ($)")

# Scatter plot of total_bill vs tip colored by smoker status
sns.scatterplot(x="total_bill", y="tip", hue="smoker", data=tips, ax=axes[1, 0], palette="Set1", alpha=0.7)
axes[1, 0].set_title("Total Bill vs Tip (by Smoker Status)", fontweight="bold")
axes[1, 0].set_xlabel("Total Bill ($)")
axes[1, 0].set_ylabel("Tip ($)")

# Correlation heatmap of numeric variables
numeric_tips = tips.select_dtypes(include=[np.number])
sns.heatmap(numeric_tips.corr(), annot=True, cmap="coolwarm", center=0, ax=axes[1, 1], fmt=".2f")
axes[1, 1].set_title("Correlation Heatmap", fontweight="bold")

plt.tight_layout()
plt.show()

**Explanation of the visualizations:**

1. **Histogram with KDE** (top-left): The histogram shows the frequency distribution of total bills in bins. The KDE (Kernel Density Estimate) curve is a smoothed version of the histogram — it estimates the probability density function. Together they show that most bills cluster around $15–20, with a right skew (some expensive meals).

2. **Box plot by day** (top-right): Box plots show the median (line), interquartile range (box = 25th to 75th percentile), whiskers (1.5× IQR), and outliers (dots). Comparing across days reveals whether tipping varies by day.

3. **Scatter plot colored by smoker** (bottom-left): Each dot is a meal. The x-axis is the bill, y-axis is the tip. Color coding by smoker status lets you visually check if smokers tip differently. The overall upward trend confirms the positive correlation.

4. **Correlation heatmap** (bottom-right): A matrix showing correlations between all numeric variables. Red = positive correlation, blue = negative. `annot=True` prints the actual values on the cells. This is a quick way to spot all relationships at once.

> **Summary of findings:** Total bill and tip are strongly correlated (~0.67). Most dining parties are size 2. Tips tend to be slightly higher on weekends (Sat/Sun). The tip percentage is roughly similar across genders. The data is right-skewed — most bills are moderate with a few expensive outliers.

---
## Exercise 4: Feature Engineering Challenge

**Objective:** Create informative features from raw data.

**Why is this important?**

Feature engineering is the process of **creating new variables from existing data** to make patterns more visible to machine learning models. It's often said that feature engineering is where the real value lies in data science — a simple model with great features often outperforms a complex model with poor features.

**Types of feature engineering we'll practice:**
- **Derived features** — combine existing columns (e.g., `total_amount = quantity × price`)
- **Temporal features** — extract useful info from dates (day of week, month, is_weekend)
- **Binning** — convert continuous values into categories (age → age_group)
- **Aggregation** — summarize data at a higher level (customer-level stats)
- **Encoding** — convert categories into numbers (one-hot encoding)
- **Scaling** — normalize numeric ranges (Min-Max scaling)

### Task 1: Create the sample e-commerce dataset

In [ ]:
np.random.seed(42)

ecommerce = pd.DataFrame({
    "order_id": range(1, 101),
    "customer_id": np.random.randint(1, 30, 100),
    "order_date": pd.date_range("2024-01-01", periods=100, freq="D"),
    "product_category": np.random.choice(["Electronics", "Clothing", "Books", "Food"], 100),
    "quantity": np.random.randint(1, 10, 100),
    "unit_price": np.random.uniform(5, 200, 100).round(2),
    "customer_age": np.random.randint(18, 70, 100),
    "is_returned": np.random.choice([0, 1], 100, p=[0.85, 0.15])
})

print(f"Shape: {ecommerce.shape}")
ecommerce.head()

**Explanation:**

We use `np.random.seed(42)` to make the random data reproducible — anyone running this code will get the exact same "random" numbers. The dataset simulates an e-commerce store with 100 orders from ~30 customers across 4 product categories.

> **Why seed = 42?** It's a convention in data science (a nod to "The Hitchhiker's Guide to the Galaxy"). Any integer works — the point is reproducibility.

### Task 2: Create order-level features

In [ ]:
# total_amount
ecommerce["total_amount"] = ecommerce["quantity"] * ecommerce["unit_price"]

# day_of_week and is_weekend
ecommerce["day_of_week"] = ecommerce["order_date"].dt.day_name()
ecommerce["is_weekend"] = ecommerce["order_date"].dt.dayofweek >= 5

# month
ecommerce["month"] = ecommerce["order_date"].dt.month

# age_group
ecommerce["age_group"] = pd.cut(
    ecommerce["customer_age"],
    bins=[0, 30, 50, 120],
    labels=["Young", "Adult", "Senior"]
)

# high_value_order
ecommerce["high_value_order"] = ecommerce["total_amount"] > 500

print("New features created:")
ecommerce[["total_amount", "day_of_week", "is_weekend", "month", "age_group", "high_value_order"]].head(10)

**Explanation of each feature:**

| Feature | How it's created | Why it's useful |
|---------|-----------------|-----------------|
| `total_amount` | `quantity × unit_price` | The raw columns alone don't tell you order value — this **derived feature** combines them |
| `day_of_week` | `.dt.day_name()` | Buying patterns differ by day (e.g., more impulse buys on Friday) |
| `is_weekend` | `.dt.dayofweek >= 5` | A simplified version — binary features are easy for models to use |
| `month` | `.dt.month` | Captures seasonality (holiday shopping in December, etc.) |
| `age_group` | `pd.cut()` with bins | **Binning** converts a continuous variable into meaningful categories. Models can sometimes learn from groups better than raw numbers |
| `high_value_order` | `total_amount > 500` | A **threshold-based binary feature** — useful for classification tasks |

**`pd.cut()`** assigns each value to a bin:
- `bins=[0, 30, 50, 120]` creates ranges: 0–30, 31–50, 51–120
- `labels=["Young", "Adult", "Senior"]` gives human-readable names to each bin

### Task 3: Create aggregated customer-level features

In [ ]:
customer_features = ecommerce.groupby("customer_id").agg(
    total_orders=("order_id", "count"),
    avg_order_value=("total_amount", "mean"),
    return_rate=("is_returned", "mean")
).round(2)

print("Customer-level features:")
customer_features.head(10)

**Explanation:**

**Customer-level aggregation** transforms order-level data into customer-level features using `groupby().agg()`:

- **`total_orders`** — count of orders per customer. Frequent buyers behave differently from one-time buyers.
- **`avg_order_value`** — mean of `total_amount`. Helps identify high-value vs budget customers.
- **`return_rate`** — mean of `is_returned` (since it's 0/1, the mean equals the proportion of returns). A customer with a 50% return rate is very different from one with 0%.

The `.agg()` method takes a dictionary mapping output column names to `(input_column, function)` tuples. This is more readable and flexible than chaining multiple `.groupby()` calls.

> **Why aggregate?** Machine learning models often need one row per entity (customer). If your data has multiple rows per customer (orders), you need to aggregate to the customer level.

### Task 4: One-hot encode product_category

In [ ]:
encoded = pd.get_dummies(ecommerce["product_category"], prefix="cat")
ecommerce = pd.concat([ecommerce, encoded], axis=1)

print("One-hot encoded columns:")
ecommerce[encoded.columns].head()

**Explanation:**

**One-hot encoding** converts a categorical column into multiple binary (0/1) columns — one per category:

```
product_category  →  cat_Books  cat_Clothing  cat_Electronics  cat_Food
Electronics          0          0             1                0
Books                1          0             0                0
Food                 0          0             0                1
```

**Why is this necessary?** Most ML algorithms only work with numbers. You can't feed the string `"Electronics"` into a model. One-hot encoding creates numeric representations without implying any ordering (unlike label encoding where Electronics=0, Clothing=1 might imply Clothing > Electronics).

- **`pd.get_dummies()`** does this in one line
- **`prefix="cat"`** adds a prefix to the new column names for clarity
- **`pd.concat()`** joins the new columns back to the original DataFrame

### Task 5: Apply Min-Max scaling to unit_price and quantity

In [ ]:
for col in ["unit_price", "quantity"]:
    min_val = ecommerce[col].min()
    max_val = ecommerce[col].max()
    ecommerce[f"{col}_scaled"] = ((ecommerce[col] - min_val) / (max_val - min_val)).round(4)

print("Scaled features:")
ecommerce[["unit_price", "unit_price_scaled", "quantity", "quantity_scaled"]].head(10)

**Explanation:**

**Min-Max Scaling** transforms values to a [0, 1] range using the formula:

$$x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

- The minimum value becomes 0
- The maximum value becomes 1
- Everything else is proportionally between 0 and 1

**Why scale features?** Many ML algorithms (e.g., KNN, SVM, neural networks) are sensitive to the scale of features. If `unit_price` ranges from 5–200 and `quantity` ranges from 1–9, the model would give disproportionate weight to `unit_price` simply because its numbers are larger. Scaling puts all features on a level playing field.

> **Note:** In practice, you'd use `sklearn.preprocessing.MinMaxScaler` which handles fitting and transforming properly for train/test splits. The manual formula here is for understanding the concept.

---
## Exercise 5: Outlier Detection and Treatment

**Objective:** Identify and handle outliers using multiple methods.

**Why is this important?**

**Outliers** are data points that are significantly different from other observations. They can be:
- **Errors** — data entry mistakes, sensor malfunctions (should be removed)
- **Legitimate extreme values** — a billionaire's income in a salary dataset (handle carefully)
- **Interesting anomalies** — fraud detection is literally finding outliers

Outliers can distort your analysis:
- They **skew the mean** (one extreme value can shift the average significantly)
- They **affect model training** — linear regression tries to fit outliers, hurting predictions for normal data
- They **inflate variance** — making standard deviation unreliable

Learning multiple detection methods helps you decide which outliers are real problems vs valuable signal.

### Task 1: Generate a dataset with known outliers

In [ ]:
np.random.seed(42)
normal_data = np.random.normal(50, 10, 200)
outliers = np.array([150, -30, 180, 200, -50])
data = pd.DataFrame({"value": np.concatenate([normal_data, outliers])})

print(f"Dataset shape: {data.shape}")
print(f"Known outliers injected: {outliers}")
data.describe()

**Explanation:**

We generate 200 values from a **normal distribution** centered at 50 with a standard deviation of 10. Most values will fall between 20–80. Then we inject 5 **known outliers** (150, -30, 180, 200, -50) that are far outside this range.

This is a controlled experiment — because we know exactly which points are outliers, we can evaluate how well each detection method identifies them.

### Task 2: Detect outliers using IQR method

In [ ]:
Q1 = data["value"].quantile(0.25)
Q3 = data["value"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

iqr_outliers = data[(data["value"] < lower_bound) | (data["value"] > upper_bound)]
print(f"IQR Method — Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Outliers found: {len(iqr_outliers)}")
print(iqr_outliers["value"].values)

**Explanation — IQR Method:**

The **Interquartile Range (IQR)** method is the most widely used outlier detection technique:

1. Calculate **Q1** (25th percentile) and **Q3** (75th percentile)
2. **IQR = Q3 - Q1** (the range of the middle 50% of data)
3. Define bounds:
   - **Lower bound = Q1 - 1.5 × IQR**
   - **Upper bound = Q3 + 1.5 × IQR**
4. Anything outside these bounds is an outlier

**Why 1.5?** This is a convention established by statistician John Tukey. For normally distributed data, the 1.5× IQR rule flags roughly the most extreme 0.7% of values. Using 3× IQR would only flag the most extreme values.

> **Advantage:** The IQR method is **robust to outliers** — unlike mean and standard deviation, quartiles aren't distorted by extreme values.

### Task 3: Detect outliers using Z-score method

In [ ]:
from scipy import stats

z_scores = np.abs(stats.zscore(data["value"]))
zscore_outliers = data[z_scores > 3]

print(f"Z-score Method (threshold=3)")
print(f"Outliers found: {len(zscore_outliers)}")
print(zscore_outliers["value"].values)

**Explanation — Z-Score Method:**

The **Z-score** measures how many standard deviations a value is from the mean:

$$z = \frac{x - \mu}{\sigma}$$

- A Z-score of 0 means the value equals the mean
- A Z-score of 3 means the value is 3 standard deviations above the mean
- Values with |Z| > 3 are considered outliers (the threshold is customizable)

**IQR vs Z-Score — when to use which?**

| Method | Pros | Cons |
|--------|------|------|
| **IQR** | Robust to outliers; works with skewed data | May flag too many in non-normal distributions |
| **Z-Score** | Easy to interpret; tunable threshold | Sensitive to outliers (mean and std are affected by them) |

> **Note:** Both methods assume a unimodal distribution. For multi-modal data (data with multiple peaks), neither method works well — you'd need more advanced techniques like DBSCAN.

### Task 4: Visual inspection using box plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].boxplot(data["value"], patch_artist=True,
                boxprops=dict(facecolor="#4ECDC4", alpha=0.7))
axes[0].set_title("Box Plot of Values", fontweight="bold")
axes[0].set_ylabel("Value")

axes[1].hist(data["value"], bins=30, color="#FF6B6B", alpha=0.7, edgecolor="black")
axes[1].axvline(lower_bound, color="blue", linestyle="--", label=f"IQR Lower ({lower_bound:.1f})")
axes[1].axvline(upper_bound, color="blue", linestyle="--", label=f"IQR Upper ({upper_bound:.1f})")
axes[1].set_title("Distribution with IQR Bounds", fontweight="bold")
axes[1].set_xlabel("Value")
axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout()
plt.show()

**Explanation — Visual Inspection:**

Visual inspection is always a good complement to statistical methods:

- **Box plot** — the dots beyond the whiskers are outliers (using the IQR rule). This gives an instant visual sense of the data's spread and extreme values.
- **Histogram with IQR bounds** — shows the full distribution with blue dashed lines marking the IQR boundaries. Values beyond these lines are outliers.

> **Best practice:** Always combine statistical methods with visual inspection. Numbers tell you *what* is an outlier; plots help you understand *why* and whether it makes sense to remove it.

### Task 5: Apply three different outlier treatments and compare

In [ ]:
# Treatment 1: Remove outliers
data_removed = data[(data["value"] >= lower_bound) & (data["value"] <= upper_bound)]

# Treatment 2: Winsorization (cap at bounds)
data_capped = data.copy()
data_capped["value"] = data_capped["value"].clip(lower=lower_bound, upper=upper_bound)

# Treatment 3: Log transform (shift to make all values positive first)
data_log = data.copy()
shift = abs(data["value"].min()) + 1
data_log["value"] = np.log1p(data["value"] + shift)

# Visualize all three treatments
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

treatments = [
    ("Removal", data, data_removed),
    ("Winsorization (Capping)", data, data_capped),
    ("Log Transform", data, data_log),
]

for i, (name, before, after) in enumerate(treatments):
    axes[i, 0].hist(before["value"], bins=30, color="#FF6B6B", alpha=0.7, edgecolor="black")
    axes[i, 0].set_title(f"Before {name}", fontweight="bold")
    axes[i, 0].set_xlabel("Value")
    axes[i, 0].set_ylabel("Frequency")

    axes[i, 1].hist(after["value"], bins=30, color="#4ECDC4", alpha=0.7, edgecolor="black")
    axes[i, 1].set_title(f"After {name} (n={len(after)})", fontweight="bold")
    axes[i, 1].set_xlabel("Value")
    axes[i, 1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

**Explanation of the three treatments:**

**1. Removal** — Simply delete outlier rows.
- **Pros:** Clean data, no distortion
- **Cons:** Reduces dataset size; if outliers are legitimate, you lose real information
- **Use when:** Outliers are clearly errors (sensor glitches, typos)

**2. Winsorization (Capping)** — Replace outliers with the nearest boundary value. Values below the lower bound become the lower bound; values above the upper become the upper.
- **Pros:** Preserves dataset size; limits extreme influence
- **Cons:** Creates artificial clusters at the boundaries
- **Use when:** You need all rows but want to reduce outlier impact

**3. Log Transform** — Apply `log(x)` to compress the scale. Large values get compressed more than small ones. We shift values first (`+ abs(min) + 1`) to ensure all values are positive (can't take log of negative numbers).
- **Pros:** Handles right-skewed data naturally; no data loss
- **Cons:** Changes the distribution shape; harder to interpret values
- **Use when:** Data is naturally right-skewed (income, prices, counts)

### Discussion

- **Removal** is simplest but reduces dataset size — best when outliers are errors, not real data points.
- **Winsorization** preserves dataset size by capping extreme values — good when you want to limit influence without losing rows.
- **Log transform** compresses the scale and reduces the impact of extreme values — best when data is naturally right-skewed.

For this dataset, **removal** is most appropriate since the outliers (150, -30, 180, 200, -50) were artificially injected and are clearly not from the same normal distribution.